In [1]:
from google.colab import drive
drive.mount("/content/drive")

import torch
print("GPU available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

PROJECT_DIR = "/content/drive/MyDrive/envsdd_project/speech"

Mounted at /content/drive
GPU available: True Tesla T4


In [2]:
import os
os.makedirs("/content/drive/MyDrive/envsdd_project", exist_ok=True)
if not os.path.isdir(PROJECT_DIR):
    !unzip -q "/content/environmental_sound_deepfake_part3_detection_models.zip" -d /content/drive/MyDrive/envsdd_project
%cd {PROJECT_DIR}
!pip install -q -r requirements.txt

/content/drive/MyDrive/envsdd_project/speech
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 14.9 MB/s eta 0:00:00


In [3]:
%cd {PROJECT_DIR}
!chmod +x run_preprocessing.sh
!./run_preprocessing.sh

/content/drive/MyDrive/envsdd_project/speech
=== EnvSDD preprocessing ===
train=1200 groups (~6000 clips)
val=300 groups (~1500 clips)
test=300 groups (~2400 clips)
total ~9900 clips, ~1237 MB

>>> fetching train (1200 groups) - logging to logs/fetch_train.log
train: 139055 rows, block=5, 27811 source groups
plan: 60 requests covering 1200 groups
  [1/60] rows=100 ok=0 skip=100 fail=0 19s
  [2/60] rows=200 ok=0 skip=200 fail=0 21s
  [3/60] rows=300 ok=0 skip=300 fail=0 23s
  [4/60] rows=400 ok=0 skip=400 fail=0 27s
  [5/60] rows=500 ok=0 skip=500 fail=0 29s
  [6/60] rows=600 ok=0 skip=600 fail=0 31s
  [7/60] rows=700 ok=0 skip=700 fail=0 34s
  [8/60] rows=800 ok=0 skip=800 fail=0 36s
  [9/60] rows=900 ok=0 skip=900 fail=0 39s
  [10/60] rows=1000 ok=0 skip=1000 fail=0 42s
  [11/60] rows=1100 ok=0 skip=1100 fail=0 44s
  [12/60] rows=1200 ok=0 skip=1200 fail=0 46s
  [13/60] rows=1300 ok=0 skip=1300 fail=0 50s
  [14/60] rows=1400 ok=0 skip=1400 fail=0 52s
  [15/60] rows=1500 ok=0 skip=1500

In [4]:
%cd {PROJECT_DIR}
!python -m src.training.smoke_test

!python -m src.training.train --model cnn          --epochs 30 --batch_size 32
!python -m src.training.train --model aasist       --epochs 30 --batch_size 32
!python -m src.training.train --model beats_aasist --epochs 15 --batch_size 32
!python -m src.training.train --model fusion       --epochs 15 --batch_size 32 \
    --cnn_checkpoint results/models/cnn_best.pt \
    --beats_aasist_checkpoint results/models/beats_aasist_best.pt

/content/drive/MyDrive/envsdd_project/speech
device: cuda
train clips: 6000, validation clips: 1500

--- cnn ---
trainable params: 19,073
one train step OK, loss=0.8449
one eval pass OK, val_eer=0.3767 (535.5s)

--- aasist ---
trainable params: 269,427
one train step OK, loss=0.6812
one eval pass OK, val_eer=0.5121 (8.9s)

--- beats_aasist ---
Downloading: "https://download.pytorch.org/torchaudio/models/wav2vec2_fairseq_base_ls960.pth" to /root/.cache/torch/hub/checkpoints/wav2vec2_fairseq_base_ls960.pth
100% 360M/360M [00:01<00:00, 248MB/s]
trainable params: 263,937
one train step OK, loss=0.7108
one eval pass OK, val_eer=0.5258 (38.4s)

--- fusion ---
trainable params: 324,227
one train step OK, loss=0.8207
one eval pass OK, val_eer=0.5004 (37.8s)

=== smoke test summary ===
  cnn             OK
  aasist          OK
  beats_aasist    OK
  fusion          OK
device: cuda
train clips: 6000, validation clips: 1500
cnn: 19,073 trainable / 19,073 total parameters
epoch   1/30  train_loss=

In [5]:
%cd {PROJECT_DIR}
!python -m src.evaluation.evaluate --compare "results/models/*_best.pt"

!zip -qr /content/envsdd_results.zip results data/metadata/manifest.csv
from google.colab import files
files.download("/content/envsdd_results.zip")

/content/drive/MyDrive/envsdd_project/speech

=== aasist (checkpoint epoch 29) ===
Overall  EER=0.1879  Acc=0.8875  F1=0.9342  ROC-AUC=0.9086  (n=2400)
              eer  n_clips   seen
generator                        
G01        0.1000      300   True
G02        0.1033      300   True
G03        0.1733      300   True
G04        0.1033      300   True
G05        0.2867      300  False
G06        0.2367      300  False
G07        0.1150      300  False
Seen (G01-G04) mean EER   : 0.1200
Unseen (G05-G07) mean EER : 0.2128

=== beats_aasist (checkpoint epoch 11) ===
Overall  EER=0.2938  Acc=0.8346  F1=0.9024  ROC-AUC=0.7830  (n=2400)
              eer  n_clips   seen
generator                        
G01        0.2167      300   True
G02        0.2783      300   True
G03        0.2467      300   True
G04        0.2283      300   True
G05        0.3117      300  False
G06        0.2700      300  False
G07        0.5100      300  False
Seen (G01-G04) mean EER   : 0.2425
Unseen (G05-G07) m

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>